### Delta and Spark

Objectives :
- Receive reproducible batches and track execution attempts.
- Prevent duplicate sales when a batch is replayed.
- Read one relevant Spark plan and justify a decision.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.window import Window
from datetime import datetime
from delta.tables import DeltaTable

In [0]:
%run ./Project_DailyFranchiseSales/00_config

In [0]:
%run ./Project_DailyFranchiseSales/00_functions

#### 4.1 - Simulate incremental ingestion 

##### Create batches A and B from splited bronze sales transactions table (splited by datetime)

In [0]:
%sql
-- Use the bronze_transactions_fact as fix snapshot (ingested by DailyFranchiseSales project)

DESCRIBE TABLE workspace.bakehouse_training.bronze_transactions_fact

In [0]:
bronze_transactions_fact = 'workspace.bakehouse_training.bronze_transactions_fact'
source_df = spark.read.table(bronze_transactions_fact)

In [0]:
# show the date range 
source_df.select(F.min('dateTime').alias('min_date'), F.max('dateTime').alias('max_date')).show(truncate=False)

In [0]:
split_date = '2024-05-15'
batch_a_df = source_df.filter(F.col('dateTime') < F.lit(split_date).cast('timestamp'))
batch_b_df = source_df.filter(F.col('dateTime') >= F.lit(split_date).cast('timestamp'))
invalid_date_df = source_df.filter(F.col('dateTime').isNull())

print(f'Batch A has {batch_a_df.count()} rows')
print(f'Batch B has {batch_b_df.count()} rows')
print(f'Invalid_date has {invalid_date_df.count()} rows')

##### Prepare empty tables in schema before running the ingestion 

In [0]:
# For the dimension tables like franchises and customers, theirs silver tables producted by DailyFranhiseSales Project can be reused
catalog_name = 'workspace'
schema_name = 'bakehouse_training'

bronze_transactions_fact_inc = (f'{catalog_name}.{schema_name}.bronze_transactions_fact_inc')
silver_transactions_fact_inc = (f'{catalog_name}.{schema_name}.silver_transactions_fact_inc')
silver_invalid_transactions_fact_inc = (f'{catalog_name}.{schema_name}.silver_invalid_transactions_fact_inc')
gold_daily_franchise_sales_inc = (f'{catalog_name}.{schema_name}.gold_daily_franchise_sales_inc')
pipeline_audit = (f'{catalog_name}.{schema_name}.pipeline_audit')

In [0]:
def create_empty_table_in_schema(df: DataFrame, save_as: str) -> None:
    (
        df.limit(0)
        .write
        .format('delta')
        .mode('ignore')
        .saveAsTable(save_as)
    )

In [0]:
# Create table schema for audit table 
audit_table_schema = StructType([
    StructField('_run_id', IntegerType(), False),
    StructField('_batch_id', StringType(), False),
    StructField('start_time', TimestampType(), False),
    StructField('end_time', TimestampType(), True),
    StructField('received_rows', IntegerType(), True),
    StructField('valid_rows', IntegerType(), True),
    StructField('invalid_rows', IntegerType(), True),
    StructField('removed_duplicates', IntegerType(), True),
    StructField('status', StringType(), False)
])

audit_df = spark.createDataFrame([], audit_table_schema)


In [0]:
# Create empty tables :

bronze_inc_template = (
    spark.read.table(bronze_transactions_fact)
    .withColumn('_run_id', F.lit(None).cast('int'))
)

# For silver_transactions_fact, there is no '_run_id' either '_batch_id', so we need to add them manually
silver_inc_template = (
    spark.read.table(silver_transactions_fact)
    .withColumn('_batch_id', F.lit(None).cast('string'))
    .withColumn('_run_id', F.lit(None).cast('int'))
)

# for bronze incremental transaction table 
create_empty_table_in_schema(bronze_inc_template, bronze_transactions_fact_inc)
# for silver incremental transaction table 
create_empty_table_in_schema(
    silver_inc_template, 
    silver_transactions_fact_inc
)
# for invalid transaction table
create_empty_table_in_schema(
    silver_inc_template,
    silver_invalid_transactions_fact_inc
)

# for gold metric table 
create_empty_table_in_schema(
    spark.read.table(gold_transactions_metrics),
    gold_daily_franchise_sales_inc
)

# for audit table 
create_empty_table_in_schema(
    audit_df,
    pipeline_audit
)


####Pipeline for batch A

##### First run : ingest batch A

In [0]:
def get_next_run_id(df: str) -> int:
    last_run_id = (
        spark.read.table(df)
        .agg(F.max('_run_id'))
        .first()[0]
    )

    if last_run_id is None:
        return 1
    else:
        return last_run_id + 1                                                                                 

run_id = get_next_run_id(pipeline_audit)

def ingest_bronze_inc(df: DataFrame, batch_id: str, run_id: int, source: str, save_as: str) -> None:
    df = (
        df
        .withColumn('_batch_id', F.lit(batch_id))
        .withColumn('_run_id', F.lit(run_id))
        .withColumn('_ingested_at', F.current_timestamp())
        .withColumn('_source', F.lit(source))
    )

    (
        df.write.mode('append').format('delta').saveAsTable(save_as)
    )


In [0]:
# Adding data into pipeline_audit table before ingesting the bronze inc table 

def start_audit(run_id: int, batch_id: str, save_as: str) -> None:
    df= spark.createDataFrame(
        [(run_id,
          batch_id,
          datetime.now(),
          None,
          None,
          None,
          None,
          None,
          'Running')],
        schema = audit_table_schema
    )
    (df.write.format('delta').mode('append').saveAsTable(save_as))
    
# Update the audit table after Silver 
def end_audit(
    audit_table: str, 
    run_id: int, 
    received_rows: int, 
    valid_rows: int,
    invalid_rows : int,
    removed_duplicates : int) -> None:

    audit_delta = DeltaTable.forName(spark, audit_table)

    audit_delta.update(
        condition = F.col('_run_id') == run_id,
        set = {
            'end_time' :F.current_timestamp(),
            'received_rows': F.lit(received_rows),
            'valid_rows': F.lit(valid_rows),
            'invalid_rows': F.lit(invalid_rows),
            'removed_duplicates': F.lit(removed_duplicates),
            'status': F.lit('Success')
        }
    )

# Prevent FAIL for audit 
def fail_audit(audit_table: str, run_id:int) -> None:
    audit_delta = DeltaTable.forName(spark, audit_table)

    audit_delta.update(
        condition=F.col('_run_id') == run_id,
        set={
            'end_time': F.current_timestamp(),
            'status': F.lit('Fail')
        }
    )


In [0]:
"""run_id = get_next_run_id(pipeline_audit)
batch_id = 'batch_A'
source = bronze_transactions_fact

start_audit(
    run_id=run_id,
    batch_id=batch_id,
    save_as=pipeline_audit
)

ingest_bronze_inc(
    df=batch_a_df,
    batch_id=batch_id,
    run_id=run_id,
    source=source,
    save_as=bronze_transactions_fact_inc
)"""

##### Silver for batch A

Reuse silver transformation functions already done 

In [0]:
"""# Read bronze inc table (only batch A)
bronze_inc_transa_df = spark.read.table(bronze_transactions_fact_inc)

# Standarize transaction table
std_inc_transa_df = standarize_transactions(bronze_inc_transa_df)

# Deduplicate
dedup_inc_transa_df, quality_inc_transa_df = check_global_quality(std_inc_transa_df, 'transactionID')

# Validate 
# For validation, customers and franchises tables are needed
customers_dim = spark.read.table(silver_customers_dim)
franchises_dim = spark.read.table(silver_franchises_dim)

checked_inc_transa_df = validate_transactions(dedup_inc_transa_df, customers_dim, franchises_dim)

# Split valid and invalid rows into 2 differents tables 
valid_inc_transa_df, invalid_inc_transa_df = split_valid_invalid(checked_inc_transa_df)"""

In [0]:
def write_silver_inc(df:DataFrame, save_as:str) -> None:
    (
        df.write
        .format('delta')
        .mode('append')
        .saveAsTable(save_as)
    )

In [0]:
"""# Write into silver inc table 
write_silver_inc(valid_inc_transa_df, silver_invalid_transactions_fact_inc)
write_silver_inc(invalid_inc_transa_df, silver_invalid_transactions_fact_inc)"""

##### Gold layer

In [0]:
"""silver_transa_inc_df = spark.read.table(silver_transactions_fact_inc)"""

In [0]:
"""# Join the sales and franchises tables
transactions_franchises_df = enrich_with_franchises(
    silver_transa_inc_df, 
    franchises_dim,
    cols_franchises = ['franchiseID', 'franchiseName', 'city', 'district', 'country', 'size'])"""

In [0]:
"""# Create the gold table 
daily_sales_franchises_metrics = (
    transactions_franchises_df
    .groupBy('franchiseID', 'franchiseName', 'transaction_date')
    .agg(F.sum('totalPrice').alias('sum_revenue'),
         F.round(F.avg('totalPrice'), 2).alias('avg_revenue'),
         F.countDistinct('customerID').alias('amount_of_customers')
         )
)
daily_sales_franchises_metrics = (
    daily_sales_franchises_metrics.withColumn(
        'rank_sum_revenue',
        F.dense_rank().over(Window.partitionBy('transaction_date').orderBy(F.desc('sum_revenue')))
    )
)"""

In [0]:
"""def write_gold_inc(df: DataFrame,save_as: str) -> None:
    (
        df.write
        .format('delta')
        .mode('append')
        .saveAsTable(save_as)
    )"""

In [0]:
"""write_gold_inc(df=daily_sales_franchises_metrics, save_as=gold_daily_franchise_sales_inc)"""

##### Update pipeline audit after silver 

In [0]:
"""# get the variables to actualize pipeline_audit
received_rows = bronze_inc_transa_df.filter(F.col('_run_id') == run_id).count()
valid_rows = valid_inc_transa_df.count()
invalid_rows = invalid_inc_transa_df.count()
removed_duplicates = std_inc_transa_df.count() - dedup_inc_transa_df.count()"""

In [0]:
"""end_audit(
    audit_table=pipeline_audit, 
    run_id=run_id, 
    received_rows=received_rows,
    valid_rows=valid_rows,
    invalid_rows=invalid_rows,
    removed_duplicates=removed_duplicates
    )"""

In [0]:
"""%sql
select*
from workspace.bakehouse_training.pipeline_audit"""

#### Once validate, wrap the pipeline process to apply to batch B

In [0]:
"""def process_batch(batch_df:DataFrame, batch_id: str) -> None:
    # 1.New execution for new run_id 
    run_id = get_next_run_id(pipeline_audit)
    print(f"Starting {batch_id} - run_id={run_id}")

    # 2. Start audit 
    start_audit(
        run_id=run_id,
        batch_id=batch_id,
        save_as=pipeline_audit
        )
    try:

        # 3. Ingest bronze 
        print("Ingest Bronze...")
        ingest_bronze_inc(
            df=batch_df,
            batch_id=batch_id,
            run_id=run_id,
            source=bronze_transactions_fact,
            save_as=bronze_transactions_fact_inc
        )

        # 4. Read Bronze
        bronze_run_df = (
        spark.read.table(bronze_transactions_fact_inc)
        .filter(F.col('_run_id') == run_id)
        )

        # 5. Silver : standarization
        print("Processing Silver...")
        std_transactions = standarize_transactions(bronze_run_df)
        # 6. Silver : deduplication
        dedup_transactions, quality = check_global_quality(
            std_transactions,
            'transactionID'
        )
        # 7.Silver : validation
        customers_dim = spark.read.table(silver_customers_dim)
        franchises_dim = spark.read.table(silver_franchises_dim)

        checked_transactions = validate_transactions(dedup_transactions, customers_dim, franchises_dim)
        # 8. Silver: Split valid and invalid rows
        valid_transa_df, invalid_transa_df = split_valid_invalid(checked_transactions)
        # 9. Silver : write 
        write_silver_inc(df=valid_transa_df, save_as=silver_transactions_fact_inc)
        write_silver_inc(df=invalid_transa_df, save_as=silver_invalid_transactions_fact_inc)

        # 10. Build + write Gold
        print("Building Gold...")
        transactions_franchises_df= enrich_with_franchises(
            valid_transa_df, 
            franchises_dim, 
            cols_franchises=['franchiseID', 'franchiseName', 'city', 'district', 'country', 'size']
        )

        daily_sales_franchises_metrics = (
        transactions_franchises_df
        .groupBy('franchiseID', 'franchiseName', 'transaction_date')
        .agg(F.sum('totalPrice').alias('sum_revenue'),
            F.round(F.avg('totalPrice'), 2).alias('avg_revenue'),
            F.countDistinct('customerID').alias('amount_of_customers')
            )
        )
        daily_sales_franchises_metrics = (
            daily_sales_franchises_metrics.withColumn(
                'rank_sum_revenue',
                F.dense_rank().over(Window.partitionBy('transaction_date').orderBy(F.desc('sum_revenue')))
            )
        )

        write_gold_inc(daily_sales_franchises_metrics, gold_daily_franchise_sales_inc)

        # 11. Update audit table
        print("Updating audit...")
        received_rows = bronze_run_df.count()
        valid_rows = valid_transa_df.count()
        invalid_rows = invalid_transa_df.count()
        removed_duplicates = std_transactions.count() - dedup_transactions.count()

        end_audit(
        audit_table=pipeline_audit, 
        run_id=run_id, 
        received_rows=received_rows,
        valid_rows=valid_rows,
        invalid_rows=invalid_rows,
        removed_duplicates=removed_duplicates
        )

        print(
        f"""{batch_id} completed - run_id={run_id}
        received  : {received_rows}
        valid     : {valid_rows}
        rejected  : {invalid_rows}
        duplicates: {removed_duplicates}
        """
        )
    except Exception as e:

            fail_audit(
                audit_table=pipeline_audit,
                run_id=run_id
            )

            print(f"{batch_id} failed - run_id={run_id}")
            print(f"Error: {e}")

            raise"""


#### For batch B

In [0]:
"""process_batch(
    batch_df=batch_b_df,
    batch_id='batch_B')"""

#### 4.2 - Data idempotence with _Merge_ in Delta

The function _.process_batch()_ can't garantee that only new or changed records loaded are processed each run, a _Merge_ statement ensures that only new rows or updates rows are inserted into the target table during an incremental data load.

##### Modify the pipeline process

In [0]:
# Modify standarize_transactions() in Project_DailyFranchiseSales/00_functions by adding '_run_id' and '_batch_id'.
def standarize_transactions(df:DataFrame) -> DataFrame :
    cols = ['transactionID', 'customerID', 'franchiseID', 'dateTime', 'product', 'quantity', 'unitPrice', 'totalPrice', '_batch_id', '_run_id']
    return (
        df.select(cols)
        .withColumn('transactionID', F.col('transactionID').cast('string'))
        .withColumn('customerID', F.col('customerID').cast('string'))
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumn('transaction_date', F.col('dateTime').cast('date'))
        .withColumn('unitPrice', F.col('unitPrice').cast('decimal(10,2)'))
        .withColumn('totalPrice', F.col('totalPrice').cast('decimal(10,2)'))
        .withColumn('calculated_total', F.col('unitPrice')*F.col('quantity').cast('decimal(10,2)'))
    )

# Apply insert-only MERGE to Silver layer
def merge_silver_inc(df: DataFrame, silver_table: str) -> None:
    silver_delta = DeltaTable.forName(spark, silver_table)
    (
        silver_delta.alias('target')
        .merge(
            df.alias('source'),
            'source.transactionID = target.transactionID',
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

# Since insert-only MERGE check the new records on key 'transactionID', what if the new record has same key but different business values ? P.S.: In general, new record having same key but different business values is a UPDATE, but in this case, it's a conflict. -> raise ValueError 
def check_silver_conflit(incoming_df: DataFrame, silver_table, key: str, business_cols: list[str]) -> None:
    
    existing_df = spark.read.table(silver_table)

    existing_incoming_joined_df = (
        incoming_df.alias('incoming')
        .join(
            existing_df.alias('existing'), 
            on=key, 
            how='inner'
        )
    )
    conflicts = None
    for col in business_cols:
        difference = ~F.col(f'incoming.{col}').eqNullSafe(F.col(f'existing.{col}'))
        if conflicts is None:
            conflicts = difference
        else:
            conflicts = conflicts | difference
    
    conflict_df = existing_incoming_joined_df.filter(conflicts)

    if conflict_df.count()>0 :
        raise ValueError(f'Conflicting business data found for existing {key}')

# Modify the saving mode of Gold Layer : append -> overwrite
def write_gold_inc(df: DataFrame, save_as: str) -> None:
    (
        df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(save_as)
    )

In [0]:
business_cols = ['transactionID', 'customerID', 'franchiseID', 'dateTime', 'product', 'quantity', 'unitPrice', 'totalPrice']

def process_batch(batch_df:DataFrame, batch_id: str) -> None:
    # 1.New execution for new run_id 
    run_id = get_next_run_id(pipeline_audit)
    print(f"Starting {batch_id} - run_id={run_id}")

    # 2. Start audit 
    start_audit(
        run_id=run_id,
        batch_id=batch_id,
        save_as=pipeline_audit
        )
    try:

        # 3. Ingest bronze 
        print("Ingest Bronze...")
        ingest_bronze_inc(
            df=batch_df,
            batch_id=batch_id,
            run_id=run_id,
            source=bronze_transactions_fact,
            save_as=bronze_transactions_fact_inc
        )

        # 4. Read Bronze
        bronze_run_df = (
        spark.read.table(bronze_transactions_fact_inc)
        .filter(F.col('_run_id') == run_id)
        )

        # 5. Silver : standarization
        print("Processing Silver...")
        std_transactions = standarize_transactions(bronze_run_df)
        # 6. Silver : deduplication
        dedup_transactions, quality = check_global_quality(
            std_transactions,
            'transactionID'
        )
        # 7.Silver : validation
        customers_dim = spark.read.table(silver_customers_dim)
        franchises_dim = spark.read.table(silver_franchises_dim)

        checked_transactions = validate_transactions(dedup_transactions, customers_dim, franchises_dim)
        # 8. Silver: Split valid and invalid rows
        valid_transa_df, invalid_transa_df = split_valid_invalid(checked_transactions)

        # 9. Silver : Check conflict between new and existing records
        check_silver_conflit(valid_transa_df, silver_transactions_fact_inc, 'transactionID', business_cols)
        
        # 10. Silver : Merge valid rows and write invalid rows 
        merge_silver_inc(df=valid_transa_df, silver_table=silver_transactions_fact_inc)
        write_silver_inc(df=invalid_transa_df, save_as=silver_invalid_transactions_fact_inc)

        # 11. Build + write Gold
        silver_transa_df = spark.read.table(silver_transactions_fact_inc)
        print("Building Gold...")
        transactions_franchises_df= enrich_with_franchises(
            silver_transa_df, 
            franchises_dim, 
            cols_franchises=['franchiseID', 'franchiseName', 'city', 'district', 'country', 'size']
        )

        daily_sales_franchises_metrics = (
        transactions_franchises_df
        .groupBy('franchiseID', 'franchiseName', 'transaction_date')
        .agg(F.sum('totalPrice').alias('sum_revenue'),
            F.round(F.avg('totalPrice'), 2).alias('avg_revenue'),
            F.countDistinct('customerID').alias('amount_of_customers')
            )
        )
        daily_sales_franchises_metrics = (
            daily_sales_franchises_metrics.withColumn(
                'rank_sum_revenue',
                F.dense_rank().over(Window.partitionBy('transaction_date').orderBy(F.desc('sum_revenue')))
            )
        )

        write_gold_inc(daily_sales_franchises_metrics, gold_daily_franchise_sales_inc)

        # 11. Update audit table
        print("Updating audit...")
        received_rows = bronze_run_df.count()
        valid_rows = valid_transa_df.count()
        invalid_rows = invalid_transa_df.count()
        removed_duplicates = std_transactions.count() - dedup_transactions.count()

        end_audit(
        audit_table=pipeline_audit, 
        run_id=run_id, 
        received_rows=received_rows,
        valid_rows=valid_rows,
        invalid_rows=invalid_rows,
        removed_duplicates=removed_duplicates
        )

        print(
        f"""{batch_id} completed - run_id={run_id}
        received  : {received_rows}
        valid     : {valid_rows}
        rejected  : {invalid_rows}
        duplicates: {removed_duplicates}
        """
        )
    except Exception as e:

            fail_audit(
                audit_table=pipeline_audit,
                run_id=run_id
            )

            print(f"{batch_id} failed - run_id={run_id}")
            print(f"Error: {e}")

            raise

In [0]:
# Before running the new version of pipeline, drop the tables created by the previous version
for t in [bronze_transactions_fact_inc, silver_transactions_fact_inc, silver_invalid_transactions_fact_inc, gold_daily_franchise_sales_inc, pipeline_audit]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

##### Apply the new pipeline with _MERGE_ included

In [0]:
# Run 1 : Batch A
process_batch(batch_df=batch_a_df, batch_id='batch_a')
# Run 2 : Batch B
process_batch(batch_df=batch_b_df, batch_id='batch_b')

In [0]:
# Create snapshots for Silver and Gold before replay batch B
silver_before_replay = spark.read.table(silver_transactions_fact_inc)
silver_before_replay.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{schema_name}.silver_before_replay_snapshot')
gold_before_replay = spark.read.table(gold_daily_franchise_sales_inc)
gold_before_replay.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{schema_name}.gold_before_replay_snapshot')

In [0]:
# Run 3 : replay Batch B again
process_batch(batch_df=batch_b_df, batch_id='batch_b')

In [0]:
# Create snapshots after replaying batch B
silver_after_replay = spark.read.table(silver_transactions_fact_inc)
silver_after_replay.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{schema_name}.silver_after_replay_snapshot')
gold_after_replay = spark.read.table(gold_daily_franchise_sales_inc)
gold_after_replay.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{schema_name}.gold_after_replay_snapshot')

##### Check the idempotence: contents in both directions with _exceptAll()_, row counts with _.count()_

In [0]:
# Read fix snapshots 
silver_before = spark.read.table(f'{catalog_name}.{schema_name}.silver_before_replay_snapshot')
silver_after = spark.read.table(f'{catalog_name}.{schema_name}.silver_after_replay_snapshot')

gold_before = spark.read.table(f'{catalog_name}.{schema_name}.gold_before_replay_snapshot')
gold_after = spark.read.table(f'{catalog_name}.{schema_name}.gold_after_replay_snapshot')

In [0]:
def check_idempotenc_after_replay(
    before_df: DataFrame,
    after_df: DataFrame,
    key: str | list[str],
    revenue_col: str,
    layer: str
) -> None:

    if isinstance(key, str):
        key = [key]

    # 1. Row count
    assert before_df.count() == after_df.count()

    # 2. Same contents in both directions
    assert before_df.exceptAll(after_df).count() == 0
    assert after_df.exceptAll(before_df).count() == 0

    # 3. Key uniqueness
    duplicate_keys = (
        after_df
        .groupBy(*key)
        .count()
        .filter(F.col('count') > 1)
    )
    assert duplicate_keys.count() == 0

    # 4. Same revenue
    revenue_before = (
        before_df
        .agg(F.sum(revenue_col).alias('revenue'))
        .first()['revenue']
    )

    revenue_after = (
        after_df
        .agg(F.sum(revenue_col).alias('revenue'))
        .first()['revenue']
    )

    assert revenue_before == revenue_after

    print(f"Idempotence checks for {layer} passed")

In [0]:
check_idempotenc_after_replay(
    before_df=silver_before,
    after_df=silver_after,
    key='transactionID',
    revenue_col='totalPrice',
    layer='Silver'
)

check_idempotenc_after_replay(
    before_df=gold_before,
    after_df=gold_after,
    key=['franchiseID', 'transaction_date'],
    revenue_col='sum_revenue',
    layer='Gold'
)

In [0]:
%sql 
SELECT *
FROM workspace.bakehouse_training.pipeline_audit

From audit table, the batch B has been ingested twice, and Silver and Gold layers stay unchanged.